In [1]:
#pip install -q python-dotenv numpy pandas scikit-learn transformers datasets torch --upgrade

In [2]:
import os
import random
import json
import numpy as np
import pandas as pd
from dotenv import load_dotenv

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from transformers import DataCollatorWithPadding
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

from sklearn.metrics import precision_recall_fscore_support, confusion_matrix, classification_report

load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

Using device: cuda


In [3]:
# Encoder model
MODEL_NAME = "answerdotai/ModernBERT-base"

# Model params
MAX_LENGTH = 6000
BATCH_SIZE = 8
EPOCHS = 10
LR = 2e-5
HEAD_HIDDEN = 384
DROPOUT_P = 0.2
WARMUP_RATIO = 0.06

EARLY_STOPPING_PATIENCE = 2
MIN_DELTA = 0.001

# headers: id,assignment_context,previous_conversation,user_query, assistant_response, irrelevancy,solution_proximality
TRAIN_CSV = "synthetic_dataset_raahul.csv"
VAL_CSV = "synthetic_dataset_openrouter.csv"
LABEL_COL = "irrelevancy"  # 1 = irrelevant, 0 = relevant

OUTPUT_DIR = "../irrelevancy_modernbert"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [4]:
def load_split(path):
    if os.path.exists(path):
        df = pd.read_csv(path)
        assert set([
            "id",
            "assignment_context",
            "user_query",
            LABEL_COL,
        ]).issubset(df.columns)
        return df

train_df = load_split(TRAIN_CSV)
val_df = load_split(VAL_CSV)

print("Train:", train_df.shape, "Val:", val_df.shape)

# train_df.head()

Train: (1540, 7) Val: (102, 7)


In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True,
    return_tensors="pt",
)


class IrrelevancyDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=MAX_LENGTH):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        assignment = str(row["assignment_context"]).strip()
        previous = str(row["previous_conversation"]).strip()
        query = str(row["user_query"]).strip()
        response = str(row["assistant_response"]).strip()

        text = (
            f"[ASSIGNMENT]\n{assignment}\n\n"
            f"[PREVIOUS CONVERSATION]\n{previous}\n\n"
            f"[CURRENT USER QUERY]\n{query}\n\n"
        )

        enc = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
        )

        item = {
            "input_ids": enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "label": 1 if str(row[LABEL_COL]).strip().lower() == "yes" else 0,
        }

        return item


train_ds = IrrelevancyDataset(train_df, tokenizer)
val_ds = IrrelevancyDataset(val_df, tokenizer)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=data_collator
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator
)

In [6]:
class IrrelevancyClassifier(nn.Module):
    def __init__(self, model_name=MODEL_NAME,
                 hidden_size=768,  # 1024 for ModernBERT-large
                 head_hidden=HEAD_HIDDEN,
                 dropout_p=DROPOUT_P,
                 num_labels=2):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.head = nn.Sequential(
            nn.Linear(hidden_size, head_hidden),
            nn.GELU(),
            nn.Dropout(dropout_p),
            nn.Linear(head_hidden, num_labels),
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_repr = outputs.last_hidden_state[:, 0, :]   # [CLS] pooling
        logits = self.head(cls_repr)
        return logits

model = IrrelevancyClassifier().to(DEVICE)
print(model)

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

[transformers] ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
decoder.bias      | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


IrrelevancyClassifier(
  (encoder): ModernBertModel(
    (embeddings): ModernBertEmbeddings(
      (tok_embeddings): Embedding(50368, 768, padding_idx=50283)
      (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=False)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0): ModernBertEncoderLayer(
        (attn_norm): Identity()
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=768, out_features=2304, bias=False)
          (Wo): Linear(in_features=768, out_features=768, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=False)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=768, out_features=2304, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_features=1152, out_features=768, bias=False)
        )
      )
      (1-21): 21 x ModernBertEncoderLa

In [7]:
label_counts = train_df[LABEL_COL].str.lower().value_counts()
print("Label counts (train):\n", label_counts)

n_total = label_counts.sum()
n_classes = 2

counts_by_id = {
    0: label_counts.get("no", 0),   # relevant
    1: label_counts.get("yes", 0),  # irrelevant
}

class_weights = torch.tensor(
    [n_total / (n_classes * max(counts_by_id[c], 1)) for c in range(n_classes)],
    dtype=torch.float,
).to(DEVICE)
print("Class weights:", class_weights)

criterion = nn.CrossEntropyLoss(weight=class_weights)

Label counts (train):
 irrelevancy
no     1086
yes     454
Name: count, dtype: int64
Class weights: tensor([0.7090, 1.6960], device='cuda:0')


In [8]:
# Training Hyperparams
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
warmup_steps = int(WARMUP_RATIO * total_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)


def run_epoch(loader, train_mode=True):
    model.train() if train_mode else model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []

    context = torch.enable_grad() if train_mode else torch.no_grad()

    with context:
        progress = tqdm(
            loader,
            desc="Training" if train_mode else "Validation",
            leave=False
        )

        for batch in progress:
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)

            if train_mode:
                optimizer.zero_grad()

            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)

            if train_mode:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=1.0
                )
                optimizer.step()
                scheduler.step()

            total_loss += loss.item() * input_ids.size(0)

            preds = torch.argmax(logits, dim=-1)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

            progress.set_postfix(loss=f"{loss.item():.4f}")

    avg_loss = total_loss / len(loader.dataset)

    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0
    )

    return avg_loss, precision, recall, f1, all_preds, all_labels


# Training Loop
best_val_f1 = -1.0
epochs_without_improvement = 0

for epoch in range(1, EPOCHS + 1):
    print(f"\nEpoch {epoch}/{EPOCHS}")

    train_loss, train_p, train_r, train_f1, _, _ = run_epoch(
        train_loader,
        train_mode=True
    )

    val_loss, val_p, val_r, val_f1, _, _ = run_epoch(
        val_loader,
        train_mode=False
    )

    print(
        f"train_loss={train_loss:.4f} "
        f"train_f1={train_f1:.4f} | "
        f"val_loss={val_loss:.4f} "
        f"val_f1={val_f1:.4f}",
        flush=True
    )

    if val_f1 > best_val_f1 + MIN_DELTA:
        best_val_f1 = val_f1
        epochs_without_improvement = 0

        torch.save(
            model.state_dict(),
            os.path.join(OUTPUT_DIR, "best_model.pt")
        )

        print(
            f"  -> new best val_f1={val_f1:.4f}, checkpoint saved.",
            flush=True
        )
    else:
        epochs_without_improvement += 1

        print(
            f"  -> no significant improvement "
            f"({epochs_without_improvement}/{EARLY_STOPPING_PATIENCE})",
            flush=True
        )

        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print(
                f"Early stopping triggered. "
                f"Best val_f1={best_val_f1:.4f}",
                flush=True
            )
            break


Epoch 1/10


Training:   0%|          | 0/193 [00:00<?, ?it/s]

Validation:   0%|          | 0/13 [00:00<?, ?it/s]

train_loss=0.2610 train_f1=0.8538 | val_loss=0.8111 val_f1=0.8795
  -> new best val_f1=0.8795, checkpoint saved.

Epoch 2/10


Training:   0%|          | 0/193 [00:00<?, ?it/s]

...

Validation:   0%|          | 0/13 [00:00<?, ?it/s]

train_loss=0.0000 train_f1=1.0000 | val_loss=0.4852 val_f1=0.9661
  -> no significant improvement (1/2)

Epoch 5/10


Training:   0%|          | 0/193 [00:00<?, ?it/s]

Validation:   0%|          | 0/13 [00:00<?, ?it/s]

train_loss=0.0000 train_f1=1.0000 | val_loss=0.4924 val_f1=0.9661
  -> no significant improvement (2/2)
Early stopping triggered. Best val_f1=0.9661


In [9]:
model.load_state_dict(torch.load(os.path.join(OUTPUT_DIR, "best_model.pt"), map_location=DEVICE))
model.eval()

val_loss, val_p, val_r, val_f1, val_preds, val_labels = run_epoch(val_loader, train_mode=False)

print(f"val loss: {val_loss:.4f}")
print(classification_report(val_labels, val_preds, target_names=["relevant", "irrelevant"], zero_division=0))

cm = confusion_matrix(val_labels, val_preds, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()
fpr = fp / (fp + tn) if (fp + tn) > 0 else float("nan")

print("Confusion matrix (rows=true, cols=pred), order=[relevant, irrelevant]:")
print(cm)
print(f"False Positive Rate (legitimate queries wrongly flagged irrelevant): {fpr:.4f}")

Validation:   0%|          | 0/13 [00:00<?, ?it/s]

val loss: 0.4768
              precision    recall  f1-score   support

    relevant       0.96      1.00      0.98        68
  irrelevant       1.00      0.91      0.95        34

    accuracy                           0.97       102
   macro avg       0.98      0.96      0.97       102
weighted avg       0.97      0.97      0.97       102

Confusion matrix (rows=true, cols=pred), order=[relevant, irrelevant]:
[[68  0]
 [ 3 31]]
False Positive Rate (legitimate queries wrongly flagged irrelevant): 0.0000


In [10]:
tokenizer.save_pretrained(OUTPUT_DIR)

config = {
    "setup_name": "IrrelevancyModernBERT (2-seg) Linear+GELU+Linear",
    "encoder": MODEL_NAME,
    "max_length": MAX_LENGTH,
    "head_hidden": HEAD_HIDDEN,
    "dropout_p": DROPOUT_P,
    "label_map": {"0": "relevant", "1": "irrelevant"},
}
with open(os.path.join(OUTPUT_DIR, "config.json"), "w") as f:
    json.dump(config, f, indent=2)

print(f"Saved tokenizer, best_model.pt, and config.json to ./{OUTPUT_DIR}/")

Saved tokenizer, best_model.pt, and config.json to ./../irrelevancy_modernbert/


In [11]:
import time

model.eval()
sample_batch = next(iter(val_loader))
input_ids = sample_batch["input_ids"][:1].to(DEVICE)
attention_mask = sample_batch["attention_mask"][:1].to(DEVICE)

# warmup
with torch.no_grad():
    for _ in range(5):
        _ = model(input_ids, attention_mask)

n_runs = 50
start = time.perf_counter()
with torch.no_grad():
    for _ in range(n_runs):
        _ = model(input_ids, attention_mask)
end = time.perf_counter()

avg_latency_ms = (end - start) / n_runs * 1000
print(f"Avg single-example inference latency: {avg_latency_ms:.2f} ms on {DEVICE}")

Avg single-example inference latency: 12.94 ms on cuda


In [12]:
test_df = load_split("organic_dataset.csv")
test_ds = IrrelevancyDataset(test_df, tokenizer)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=data_collator
)

In [13]:
model.load_state_dict(torch.load(os.path.join(OUTPUT_DIR, "best_model.pt"), map_location=DEVICE))
model.eval()

test_loss, test_p, test_r, test_f1, test_preds, test_labels = run_epoch(test_loader, train_mode=False)

print(f"test loss: {test_loss:.4f}")
print(classification_report(test_labels, test_preds, target_names=["relevant", "irrelevant"], zero_division=0))

cm = confusion_matrix(test_labels, test_preds, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()
fpr = fp / (fp + tn) if (fp + tn) > 0 else float("nan")

print("Confusion matrix (rows=true, cols=pred), order=[relevant, irrelevant]:")
print(cm)
print(f"False Positive Rate (legitimate queries wrongly flagged irrelevant): {fpr:.4f}")

Validation:   0%|          | 0/3 [00:00<?, ?it/s]

test loss: 2.6605
              precision    recall  f1-score   support

    relevant       0.82      0.64      0.72        14
  irrelevant       0.38      0.60      0.46         5

    accuracy                           0.63        19
   macro avg       0.60      0.62      0.59        19
weighted avg       0.70      0.63      0.65        19

Confusion matrix (rows=true, cols=pred), order=[relevant, irrelevant]:
[[9 5]
 [2 3]]
False Positive Rate (legitimate queries wrongly flagged irrelevant): 0.3571
